# Type narrowing and mypy integration

This notebook walks through **type narrowing** — how a static type checker like mypy refines a broad type into a narrower one as it follows control flow — and shows how to drive mypy from a notebook to inspect what it infers. This combines the Static Type Checking concept with the mypy tool: the same checks mypy runs in CI are run here, cell by cell, so the narrowing behavior is visible instead of abstract.

The cells write small scratch modules to a temp dir and run mypy against them, so each experiment is a real type-check rather than a hypothetical.

> Requires mypy installed in the kernel environment: `python -m pip install mypy`.

In [ ]:
# last_verified: 2026-08-16 · mypy 2.3.1
import importlib.util
import subprocess
import sys
import tempfile
from pathlib import Path

# mypy drives everything below; abort early with a clear message if it's missing.
if importlib.util.find_spec("mypy") is None:
    raise SystemExit("mypy not found — install it with `python -m pip install mypy`")

WORK = Path(tempfile.mkdtemp(prefix="narrowing-"))
print("scratch dir:", WORK)


def run_mypy(text: str, name: str, *extra: str) -> None:
    """Write `text` to a scratch module and type-check it with mypy."""
    path = WORK / name
    path.write_text(text)
    proc = subprocess.run(
        [sys.executable, "-m", "mypy", *extra, str(path)],
        capture_output=True,
        text=True,
    )
    print(f"--- mypy {name} {' '.join(extra).strip()} (exit {proc.returncode}) ---")
    print((proc.stdout + proc.stderr).strip() or "clean pass, no findings")

## What type narrowing is

Type narrowing is the checker tightening the type of a value based on what the code knows at a given point. The classic trigger is an `isinstance` check: when mypy sees `if isinstance(value, int):`, it narrows `value` from `object` to `int` inside that branch. The same happens with `is None` on an `Optional`, with truthiness tests, and when a value is reassigned to a more specific type.

Without narrowing, every `isinstance` dispatch on a broad type would force you to cast manually — with it, the branch body is statically safe all by itself.

In [ ]:
SRC = '''
def describe(value: object) -> str:
    if isinstance(value, int):
        reveal_type(value)          # narrowed to int inside this branch
        return f"int: {value}"
    if isinstance(value, str):
        reveal_type(value)          # narrowed to str
        return f"str: {value}"
    return f"other: {value!r}"
'''

run_mypy(SRC, "narrow_isinstance.py")

`reveal_type(value)` is why this works so well as a notebook exercise: it makes mypy print the type it infers for an expression at that exact spot. The two `note:` lines show `object` getting narrowed to `int` then `str` as the `isinstance` chain narrows the parameter.

> `reveal_type` produces no runtime value — mypy interprets it as a debugging print. Remove it before shipping code: leaving it in a module that actually runs makes it an undefined-name error at runtime.

## Narrowing a union and `Optional`

Narrowing works on unions too. With `Optional[str]`, mypy tracks that `None` is impossible after a `name is None` early-return, so the rest of the function sees a plain `str`. This is the same mechanism used to safely work with `Optional` values throughout a codebase — connect the dots back to the primer's `Optional[str]` term: the checker widens the type when it's declared, and narrows it back when the code proves the value isn't `None`.

In [ ]:
SRC = '''
from typing import Optional

def greet(name: Optional[str]) -> str:
    if name is None:
        reveal_type(name)           # still Optional[str] here
        return "hello, stranger"
    reveal_type(name)               # narrowed to str past the None guard
    return f"hello, {name}"
'''

run_mypy(SRC, "narrow_optional.py")

## Narrowing by reassignment

The less obvious form of narrowing is **by reassignment**: when a variable's value is replaced by something with a more specific type, mypy adopts the new type. It is *not* free license to rebind to an unrelated type — `n: int = 1; n = "x"` still errors, because the two types have no subtype relationship.

In [ ]:
SRC = '''
from typing import Sequence

def total(sizes: Sequence[int]) -> int:
    reveal_type(sizes)              # Sequence[int] — the declared param type
    concrete = list(sizes)          # reassignment to a more specific builtin
    reveal_type(concrete)           # narrowed to list[int]
    return sum(concrete)


def broken() -> None:
    n: int = 1
    n = "x"                         # error: not a compatible narrowing
'''

run_mypy(SRC, "narrow_reassign.py")

yields a single error: the `broken` reassignment. The `narrowing` in `total` is fine because `list[int]` is a subtype of `Sequence[int]` — the value got a *more specific* type, which is exactly the direction narrowing is allowed to go.

## The silent-pass trap and `--check-untyped-defs`

There's a real gap between "mypy passed" and "mypy checked everything": **unannotated functions are not type-checked at all**. An untyped def's parameters are treated as `Any`, so body errors that depend on a concrete parameter type sail through. This connects the narrowing concept back to the mypy tool's configuration — unless the project opts into checking untyped code, the narrowing-based checks never fire on those functions.

In [ ]:
SRC = '''
def concat(parts):
    return "(" + parts.split() + ")"
'''

print("== default run — untyped def, unchecked body ==")
run_mypy(SRC, "silent.py")
print()
print("== same file annotated — the error surfaces ==")
run_mypy(SRC.replace("def concat(parts):", "def concat(parts: str) -> str:"), "silent_annotated.py")

Same body, two very different verdicts. The untyped version passes silently even though `"(" + parts.split() + ")"` adds a `str` to a `list[str]`; annotating the signature makes mypy re-infer `parts.split()` and flag the operator mismatch. That's the narrowing concept in reverse — the check only fires once mypy actually knows the parameter type. This is why real projects run mypy with typed-signature enforcement, not just on whatever slips through unannotated.

In [ ]:
SRC = '''
def tally(items):
    total = 0
    for it in items:
        total += it
    label = "sum"
    label = total       # rebind str to int inside an untyped body
    return label
'''

print("== default run — untyped def, unchecked body ==")
run_mypy(SRC, "untyped.py")
print()
print("== --check-untyped-defs — body is now checked ==")
run_mypy(SRC, "untyped.py", "--check-untyped-defs")

`--check-untyped-defs` is the config-level answer to the silent pass: it type-checks the *bodies* of untyped functions, so rebindings and other body-level errors finally surface. Run it once on a legacy module and the previously-invisible `Any` body mistakes show up. (mypy 2.3.1 used throughout this notebook; the flag is `check_untyped_defs` in a `[tool.mypy]` config, so it can be made project-wide rather than CLI-only.)

## Integration trap: `Any`-poisoning

Narrowing only works when mypy knows the type it's narrowing. Any value that enters as `Any` — an unannotated parameter, an untyped function's return, an untyped `__init__` writing `self.x`, or an `import` mypy couldn't resolve — acts like a contagion: everything it touches is treated as `Any` too, and narrowing can never engage. The common fix mirrors the reassignment narrowing above: annotate the boundary (the parameter, the return, or the `self` attribute) so a concrete type exists to narrow.

In [ ]:
SRC = '''
def loader():
    return []       # no return annotation -> mypy infers Any


items = loader()    # items is Any
reveal_type(items)  # Any-poisoned: mypy cannot narrow or check items

empty: list[str] = []    # fix: annotate the empty collection
reveal_type(empty)       # list[str] — narrowable from here
'''

run_mypy(SRC, "any_poison.py")

The reveal prints make the difference visible: `items` is, and stays, `Any`, while the annotated `empty` is `list[str]`. For an untyped `loader()`, narrowing could never apply to anything downstream without first annotating `loader`'s return.

## Narrowing vs variance: containers are invariant

One last interaction that surprises people: even after narrowing, **mutable generics are invariant**. A `list[Dog]` is *not* a `list[Animal]`, no matter how many narrowing steps you take, because the list is writable. The escape hatch is to read with a covariant type like `Sequence[Animal]` instead — passing the collection around for reading loses nothing and satisfies the checker.

This is the multi-concept link: narrowing is about single *values* along a control-flow path, while variance is about how whole *container types* relate to each other. Both show up the moment you combine static type checking with real data structures.

In [ ]:
SRC = '''
from typing import List, Sequence

class Animal: ...

class Dog(Animal): ...


def feed(pets: List[Animal]) -> None:
    pass


dogs: List[Dog] = [Dog()]
feed(dogs)          # error: List is invariant

def watch(pets: Sequence[Animal]) -> None:
    pass

watch(dogs)         # OK: Sequence is covariant (read-only view)
'''

run_mypy(SRC, "variance.py")

`feed(dogs)` fails because a `List[Dog]` is not assignable to `List[Animal]`, while `watch(dogs)` passes because `Sequence` accepts the narrowing relationship. The invariant call is the one that errors — mypy refuses to pretend a writable `Dog` list is an `Animal` list.

## What this comparison established

Running mypy from a notebook turns narrowing from a static concept into a sequence of observable steps: `reveal_type` shows the type at each program point, `--check-untyped-defs` exposes the functions mypy was quietly skipping, and the variance error shows where even 'correct' narrowing is rejected. The same checks, invoked as `python -m mypy` in CI, are what this notebook runs — the concept isn't just a nicety the checker offers; it's the mechanism that makes `isinstance`/`Optional`/reassignment code safe to write.